In [1]:
from frap_utils import *

# Pipeline of analysis
* Check the .csv files with MFI of 3 ROIs generated in FIJI. If needed, change the Time scale and trim dataframes.
* Generate the .cvs file containing the information about MFI of each nucleus in the sample.
* Data filtration based on the quality of fitting ($R^2$).
* Concatenete the data from EasyFRAP with MFI nucleus.
* Spearman correlation analysis between the nuclei MFI and Mobile fraction (%) and between nuclei MFI and and Time of recovery (sec)
* Visualization (boxplots) and statistical analysis.

## Import data
* Read .csv files in the loop. Print the length of each file and its path.
*  Check the length of .csv file. Trim if needed.

In [ ]:
# Paths
'''Name of the sample'''
sample_name = "100uM_As_2h"

'''
Folder containing .csv files:
- Column 1: Time (seconds)
- Columns 2–4: MFI values from ROIs (for EasyFRAP analysis)

Each .csv file corresponds to a single cell.
'''
path = f"/mnt/c/users/elopatukhin/Desktop/Data_processing/060326_U2OS_FRAP/{sample_name}"
folder = check_dir_exists(path, ensure_dir = False)

In [ ]:
# Import data and create list of dataframes
"""
Load input files into a list of DataFrames.

Requirements:
- All files must have the same number of rows and exactly 4 columns.
- If row counts differ, trim the data accordingly (see below).
- If a file does not have 4 columns, validate and correct the input data.
"""
dfs = {}  # list of DataFrames

for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    dfs[file.stem] = df
    print(f"Path: {file}, number of rows: {df.shape[0]}, number of columns: {df.shape[1]}")

print(f"Loaded {len(dfs)} files.")

# Check data
df

### *Optional: Fix timescale*


In [ ]:
output_dir = check_dir_exists(folder.joinpath("fixed_timescale"), ensure_dir = True)

for name, df in dfs.items():
    time = 0
    nframes = len(df)
    time_factors = [(67, 0.659), (float("inf"), 5)]

    for n in range(1, nframes + 1):
        for nframe, sec in time_factors:
            if n < nframe:
                time += sec
                df.loc[n-1, "Time"] = time
                break
            else:
                continue

    df.to_csv(output_dir / f"{name}.csv", index=False) # save file

            
print(f"Fixation of timescale in {len(dfs)} dataframes")

# Check data
df.head(20)

### *Optional: trim dataframes. Export data*

In [ ]:
# Variables
dfs_new = {}  # list of DataFrames
N = 85 # keep this number of frames

output_dir = check_dir_exists(folder.joinpath("trimmed"), ensure_dir = True)

# Loop
for name, df in dfs.items():
    df_trimmed = df.iloc[:N].copy() # copy and trim dataframe

    # Fill the dictionary
    if df_trimmed.shape[1] == 4: # check number of cols in the df. Must be 4 (Time and 3 ROIs)
        dfs_new[name] = df_trimmed
        df_trimmed.to_csv(output_dir / f"{name}_trimmed.csv", index=False) # save file

    else:
        print(f"Error in the dataframe of sample: {name}: number of ROIs is: {df_trimmed.shape[1]}")

print(f"{len(dfs_new)} trimmed dataframes are saved into the directory: {output_dir}.")

### Generate dataframe containing the MFI of each nucleus before bleaching

In [ ]:
# Variables
nplateu = 15 # number of frames before bleaching
nucleus_ROI = 3 # number of column with data from the ROI corresponding the whole nucleus

# Initialize empty list to store data
MFI_nuclei_rows = []

# Loop
for name, df in dfs.items():
    mean_value = np.mean(df.iloc[:nplateu, nucleus_ROI])
    MFI_nuclei_rows.append({
            "filename": name,
            "mean_intensity": mean_value
        })
    
# Create the dataframe with MFI of nuclei before bleaching and save it
MFI_nuclei = pd.DataFrame(MFI_nuclei_rows)

print(f"Calculation of MFI of nuclei of {sample_name} sample.")

# Check data
MFI_nuclei.head()

# Run EasyFRAP web-tool
Download results in the EasyFRAP folder

In [ ]:
# Paths
# EasyFRAP folder name
easyfrap_folder_name = "EasyFRAP"

# Path to the EasyFRAP folder
easyfrap_folder_path = check_dir_exists(folder.joinpath(easyfrap_folder_name), ensure_dir = True)

# Path to the file with EasyFRAP data
easyfrap_file_path = check_file_exists(easyfrap_folder_path.joinpath(f"easyFRAP_Final_Data.xlsx"))

### Concatenate data of MFI of nuclei and EasyFRAP data. Export data.

In [ ]:
# Import EasyFRAP data as a dataframe and processe it

# Skip first 9 rows and last 2 rows
easyfrap = pd.read_excel(easyfrap_file_path, engine="calamine", skiprows=12, header=0)
easyfrap = easyfrap.iloc[:-2]
easyfrap['filename'] = easyfrap['filename'].apply(lambda x: x.split(".")[0]) # fix the filename
#easyfrap['filename'] = easyfrap['filename'].apply(lambda x: x.split(".")[0][:-8]) # fix the filename, delete the word "_trimmed"

print(f"EasyFRAP dataframe of {sample_name} sample containes {len(easyfrap)} rows.")
easyfrap.head()

In [ ]:
# Concatenate EasyFRAP data and MFI nuclei data
df_merged = pd.merge(MFI_nuclei, easyfrap, on="filename", how="right")

# Save data
df_merged_path = easyfrap_file_path.with_name(f"{easyfrap_file_path.stem}_{sample_name}_MFI.csv")
df_merged.to_csv(df_merged_path, index=False) # save data

print(f"Data are saved in the path: {df_merged_path}")

# Check data
df_merged

### Data filtration. Export data
Filtration of data based on the quality of fitting

In [ ]:
threshold_R = 0.8
df_filtered = df_merged[df_merged["R square"] > threshold_R]

print(f"Keep {len(df_filtered)} out of {len(df_merged)} rows.")

# Save filtreted data
df_filtered_path = easyfrap_file_path.with_name(f"{easyfrap_file_path.stem}_{sample_name}_MFI_filtrated.csv")
df_filtered.to_csv(df_filtered_path, index=False) 

print(f"Data are saved in the path: {df_filtered_path}")

# Check data
df_filtered

### Spearman correlation analysis. Export image

In [ ]:
# Clean data from NAs
clean = df_filtered[["mean_intensity", "Mobile Fraction", 'T-half']].dropna()
print(f"Keep {len(clean)} out of {len(df_filtered)} rows after cleaning.")

# Calculation of correlation coefficients and p-values
corr1, p1 = spearmanr(clean["mean_intensity"], clean["Mobile Fraction"])
corr2, p2 = spearmanr(clean["mean_intensity"], clean["T-half"])

# Scatterplots
fig, ax = plt.subplots(1, 2, figsize=(8, 4), dpi=150)
fig.suptitle(f"Relationship between Nuclei MFI and FRAP Parameters for ORC1 {sample_name}", fontsize=14)

ax[0].scatter(df_filtered["mean_intensity"], df_filtered["Mobile Fraction"], alpha=0.6)
ax[0].set_xlabel("Nuclei MFI")
ax[0].set_ylabel("Mobile Fraction, %")
ax[0].set_title(f"Nuclei MFI vs Mobile Fraction\nr = {corr1:.3f}, p-value = {p1:.3f}")

ax[1].scatter(df_filtered["mean_intensity"], df_filtered["T-half"], alpha=0.6)
ax[1].set_xlabel("Nuclei MFI")
ax[1].set_ylabel("Recovery Time, s")
ax[1].set_title(f"Nuclei MFI vs Recovery Time\nr = {corr2:.3f}, p-value = {p2:.3f}")

plt.tight_layout()

# Save image
plt.savefig(easyfrap_folder_path/f"correlation_plot_{sample_name}.png", dpi=300, bbox_inches="tight")
print(f"Plot is saved in the directory: {easyfrap_folder_path}.")

# Show image
plt.show()

### *Optional: Boxplots for 1 samples*

In [ ]:
beautiful_boxplot(
    df_list =
        [
        df_filtered['Mobile Fraction']
        ],
    labels =
        [
        sample_name
        ],
    ylabel="mean_intensity",
    xlabel=None,
    title=None,
    log_scale=False,
    colors=None,         # list of colors per box (optional)
    dot_size=20,
    jitter=0.06,
    figsize=(4.8, 4.2),
    dpi=200,
    show=True
);